# 레슨 01 — 실습 문제 정답지

> 🔒 **교사·관리자 전용. 학생에게 배포 금지.**
> 학생에게는 `mission.ipynb` 만 공유한다. 이 파일은 채점·강의 준비용이다.

각 문제의 모범 답안, 기대 출력, 왜 이 코드가 정답인지, 자주 보이는 오답 패턴을 정리했다. 학생 답안은 출력값만 맞는지보다 **배열을 올바르게 만들고, 마스크와 통계 함수를 의도대로 사용했는지**를 함께 본다.

## 0. 환경 셀

In [ ]:
import os
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/01/data"
else:
    DATA_BASE = "./data"

print("numpy:", np.__version__, "| data base:", DATA_BASE)

---

## 문제 1 정답 — 시험 점수 기본 요약

In [ ]:
scores = np.loadtxt(f"{DATA_BASE}/scores.csv", delimiter=",", skiprows=1)

n = scores.size
mean = scores.mean()
median = np.median(scores)
std = scores.std()
high = int(scores.max())
low = int(scores.min())

print(f"학생 수: {n}명")
print(f"평균: {mean:.2f}점")
print(f"중앙값: {median:.2f}점")
print(f"표준편차: {std:.2f}")
print(f"최고/최저: {high} / {low}")

### 왜 이 코드가 정답인지

`scores.csv` 는 첫 줄에 `score` 헤더가 있으므로 `skiprows=1` 로 헤더를 건너뛰어야 숫자 배열이 만들어진다. `scores.size` 는 배열 안 원소 수이므로 학생 수와 같다. 평균, 중앙값, 표준편차, 최댓값, 최솟값은 1차원 점수 배열 전체를 요약하는 기본 통계다. 이 문제의 목적은 CSV를 NumPy 배열로 바꾸고, 배열 메서드와 NumPy 함수를 구분해서 쓰는 것이다.

### 기대 출력

```
학생 수: 200명
평균: 75.61점
중앙값: 77.00점
표준편차: 12.39
최고/최저: 100 / 35
```

### 채점 포인트

- `skiprows=1` 누락 시 헤더 때문에 `ValueError`가 난다.
- `np.mean(scores)` 처럼 NumPy 함수로 풀어도 정답이다.
- 표준편차는 NumPy 기본값 `ddof=0` 기준으로 본다. `ddof=1`로 계산한 학생은 개념을 설명하면 부분 통과 가능하다.

---

## 문제 2 정답 — 90점 이상과 60점 미만 찾기

In [ ]:
high_mask = scores >= 90
low_mask = scores < 60

high_count = high_mask.sum()
low_count = low_mask.sum()
high_rate = high_count / scores.size * 100
low_rate = low_count / scores.size * 100

print(f"90점 이상: {high_count}명 ({high_rate:.1f}%)")
print(f"60점 미만: {low_count}명 ({low_rate:.1f}%)")

### 왜 이 코드가 정답인지

`scores >= 90` 은 각 점수마다 조건을 검사해 True/False 배열을 만든다. Boolean 배열에서 `True`는 1처럼 더해지므로 `.sum()` 으로 조건을 만족한 학생 수를 구할 수 있다. 비율은 조건을 만족한 수를 전체 학생 수로 나누고 100을 곱하면 된다. 이 문제는 NumPy 마스크가 필터링과 개수 세기에 모두 쓰인다는 점을 확인한다.

### 기대 출력

```
90점 이상: 25명 (12.5%)
60점 미만: 21명 (10.5%)
```

### 자주 보이는 오답

```text
scores >= 90.sum()
```

위 코드는 `90.sum()` 을 먼저 해석하려고 해서 잘못된다. 비교식 전체를 만든 뒤 마스크에 `.sum()` 을 붙여야 한다.

---

## 문제 3 정답 — 점수의 사분위수와 IQR

In [ ]:
q1, q2, q3 = np.quantile(scores, [0.25, 0.50, 0.75])
iqr = q3 - q1

print(f"Q1: {q1:.2f}")
print(f"Q2: {q2:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")

### 왜 이 코드가 정답인지

`np.quantile` 은 정렬된 데이터에서 지정한 위치의 값을 계산한다. 25%, 50%, 75% 위치를 한 번에 넣으면 Q1, Q2, Q3가 같은 기준으로 계산된다. Q2는 중앙값과 같은 의미이며, IQR은 중간 50% 데이터의 폭을 나타낸다. 점수 분포가 얼마나 퍼져 있는지 평균과 표준편차 말고도 확인할 수 있게 하는 문제다.

### 기대 출력

```
Q1: 67.75
Q2: 77.00
Q3: 85.00
IQR: 17.25
```

### 채점 포인트

- `np.percentile(scores, [25, 50, 75])` 로 풀어도 정답이다.
- Q2와 문제 1의 중앙값이 같은지 비교하게 하면 분위수 개념을 연결하기 좋다.

---

## 문제 4 정답 — 점수 등급 분포 만들기

In [ ]:
mask_a = scores >= 90
mask_b = (scores >= 80) & (scores < 90)
mask_c = (scores >= 70) & (scores < 80)
mask_d = (scores >= 60) & (scores < 70)
mask_f = scores < 60

a = mask_a.sum()
b = mask_b.sum()
c = mask_c.sum()
d = mask_d.sum()
f = mask_f.sum()
total = a + b + c + d + f

print(f"A: {a}명")
print(f"B: {b}명")
print(f"C: {c}명")
print(f"D: {d}명")
print(f"F: {f}명")
print(f"합계: {total}명")
assert total == scores.size

### 왜 이 코드가 정답인지

등급 구간은 서로 겹치면 안 되고 빠지는 점수도 없어야 한다. 그래서 B~D 구간은 항상 아래 경계는 포함하고 위 경계는 제외하는 방식으로 작성한다. `assert total == scores.size` 는 다섯 등급을 합쳤을 때 전체 학생 수와 같은지 확인하므로, 구간 경계 실수를 잡아준다. 이 문제의 핵심은 복수 조건을 `&` 와 괄호로 정확히 묶는 것이다.

### 기대 출력

```
A: 25명
B: 57명
C: 61명
D: 36명
F: 21명
합계: 200명
```

### 자주 보이는 오답

```text
mask_b = (scores >= 80) & (scores <= 90)
```

90점이 A와 B에 동시에 들어가므로 합계가 200보다 커진다. 경계값이 겹치지 않게 `scores < 90` 으로 써야 한다.

---

## 문제 5 정답 — 평균에서 멀리 떨어진 점수 찾기

In [ ]:
z = (scores - scores.mean()) / scores.std()
far_mask = np.abs(z) >= 2
far_indices = np.where(far_mask)[0]
far_scores = scores[far_mask]

print(f"평균에서 2표준편차 이상 떨어진 학생 수: {far_mask.sum()}명")
print("인덱스:", far_indices.tolist())
print("점수:", far_scores.astype(int).tolist())

### 왜 이 코드가 정답인지

z-score는 각 점수가 평균에서 표준편차 몇 개만큼 떨어져 있는지를 나타낸다. `(scores - mean) / std` 는 배열 전체에 한 번에 적용되므로 모든 학생의 z-score가 만들어진다. `np.abs(z) >= 2` 는 평균보다 높든 낮든 2표준편차 이상 떨어진 값을 모두 잡는다. `np.where` 는 그 학생의 위치를, `scores[far_mask]` 는 실제 점수를 보여준다.

### 기대 출력

```
평균에서 2표준편차 이상 떨어진 학생 수: 7명
인덱스: [13, 151, 152, 154, 164, 169, 179]
점수: [44, 35, 48, 41, 38, 50, 47]
```

### 채점 포인트

- `z >= 2` 만 쓰면 낮은 이상치가 빠진다. 반드시 절댓값을 써야 한다.
- 인덱스는 학생 번호가 아니라 배열 위치다. 수업에서는 0부터 시작하는 위치라는 점을 다시 짚는다.

---

## 문제 6 정답 — 90일 걸음 데이터 기본 요약

In [ ]:
raw_steps = np.loadtxt(f"{DATA_BASE}/daily_steps.csv", delimiter=",", skiprows=1)
day = raw_steps[:, 0].astype(int)
steps = raw_steps[:, 1]

max_idx = np.argmax(steps)
min_idx = np.argmin(steps)

print(f"총 일수: {steps.size}일")
print(f"총 걸음: {int(steps.sum()):,}")
print(f"일평균: {steps.mean():,.0f} 보")
print(f"표준편차: {steps.std():,.0f}")
print(f"최대일: {day[max_idx]}일차, {int(steps[max_idx]):,}보")
print(f"최소일: {day[min_idx]}일차, {int(steps[min_idx]):,}보")

### 왜 이 코드가 정답인지

`daily_steps.csv` 는 `day,steps` 두 열로 되어 있다. `raw_steps[:, 0]` 은 모든 행의 첫 번째 열, `raw_steps[:, 1]` 은 모든 행의 두 번째 열을 뜻한다. `np.argmax` 와 `np.argmin` 은 최댓값과 최솟값의 위치를 돌려주므로, 그 위치로 `day` 와 `steps` 를 동시에 조회하면 "몇 일차에 몇 보"였는지 알 수 있다.

### 기대 출력

```
총 일수: 90일
총 걸음: 745,701
일평균: 8,286 보
표준편차: 2,237
최대일: 5일차, 11,454보
최소일: 82일차, 2,100보
```

### 자주 보이는 오답

```text
print(max_idx)
```

`max_idx` 는 위치 인덱스일 뿐 실제 일자가 아니다. 실제 일자는 `day[max_idx]` 로 꺼내야 한다.

---

## 문제 7 정답 — 10,000보 달성일과 5,000보 미만일

In [ ]:
goal_mask = steps >= 10000
low_mask = steps < 5000

goal_count = goal_mask.sum()
low_count = low_mask.sum()
goal_rate = goal_count / steps.size * 100
low_days = day[low_mask].tolist()

print(f"10,000보 이상: {goal_count}일 ({goal_rate:.1f}%)")
print(f"5,000보 미만: {low_count}일")
print(f"5,000보 미만 일자: {low_days}")

### 왜 이 코드가 정답인지

걸음 수 목표 달성 여부는 하루마다 True/False로 판단할 수 있다. `steps >= 10000` 은 90일 전체의 목표 달성 마스크를 만들고, `.sum()` 은 목표를 달성한 날 수를 구한다. 낮은 활동일은 같은 방식으로 `steps < 5000` 으로 잡는다. 일자 리스트는 걸음 배열이 아니라 `day` 배열에 같은 마스크를 적용해야 한다.

### 기대 출력

```
10,000보 이상: 24일 (26.7%)
5,000보 미만: 11일
5,000보 미만 일자: [7, 21, 34, 35, 42, 49, 62, 70, 76, 82, 90]
```

### 채점 포인트

- `steps[low_mask]` 를 출력하면 낮은 걸음 수가 나오고, `day[low_mask]` 를 출력하면 낮은 활동일의 날짜가 나온다.
- 학생이 비율을 0.267처럼 출력해도 의미는 맞다. 다만 보고서 형태에서는 `%`가 더 읽기 좋다.

---

## 문제 8 정답 — 월별 걸음 평균 비교

In [ ]:
month1 = steps[:30]
month2 = steps[30:60]
month3 = steps[60:90]

m1 = month1.mean()
m2 = month2.mean()
m3 = month3.mean()

if m3 > m1 + 100:
    trend = "증가"
elif m3 < m1 - 100:
    trend = "감소"
else:
    trend = "유지"

print(f"1개월 평균: {m1:,.0f} 보")
print(f"2개월 평균: {m2:,.0f} 보")
print(f"3개월 평균: {m3:,.0f} 보")
print(f"1개월 대비 3개월 변화: {trend}")

### 왜 이 코드가 정답인지

90일 데이터를 30일씩 자르면 3개월처럼 비교할 수 있다. 슬라이싱에서 `steps[:30]` 은 0~29 위치, `steps[30:60]` 은 30~59 위치, `steps[60:90]` 은 60~89 위치를 뜻한다. 평균을 비교할 때 1~2보 차이까지 변화로 보면 과민하므로, 예시 답안은 100보 이상 차이가 날 때만 증가/감소로 판단한다.

### 기대 출력

```
1개월 평균: 8,604 보
2개월 평균: 8,298 보
3개월 평균: 7,955 보
1개월 대비 3개월 변화: 감소
```

### 채점 포인트

- 학생이 100보 기준 없이 단순히 `m3 > m1` 로 판단해도 통과 가능하다.
- 핵심은 30일 단위 슬라이싱과 평균 비교가 맞는지다.

---

## 문제 9 정답 — 평일과 주말 걸음 비교

In [ ]:
dow = (day - 1) % 7
weekend_mask = (dow == 5) | (dow == 6)
weekday_mask = ~weekend_mask

weekday_avg = steps[weekday_mask].mean()
weekend_avg = steps[weekend_mask].mean()
diff = weekday_avg - weekend_avg

print(f"평일: {weekday_mask.sum()}일, 평균 {weekday_avg:,.0f} 보")
print(f"주말: {weekend_mask.sum()}일, 평균 {weekend_avg:,.0f} 보")
print(f"평일 - 주말: {diff:+,.0f} 보")

### 왜 이 코드가 정답인지

`(day - 1) % 7` 은 1일차를 0으로 맞춘 뒤 7로 나눈 나머지를 구한다. 그러면 5와 6을 주말로 볼 수 있다. 주말 마스크의 반대인 `~weekend_mask` 는 평일 마스크다. 두 마스크를 `steps` 에 적용하면 평일과 주말의 걸음 수만 따로 뽑혀 각각 평균을 계산할 수 있다.

### 기대 출력

```
평일: 65일, 평균 9,399 보
주말: 25일, 평균 5,392 보
평일 - 주말: +4,007 보
```

### 자주 보이는 오답

```text
weekend_mask = dow == 5 | dow == 6
```

괄호가 없어서 의도와 다르게 계산된다. 배열 조건은 항상 `(조건1) | (조건2)` 형태로 감싸야 한다.

---

## 문제 10 정답 — 90일 기온 기본 통계

In [ ]:
raw_temp = np.loadtxt(f"{DATA_BASE}/temperatures.csv", delimiter=",", skiprows=1)
temp_day = raw_temp[:, 0].astype(int)
temp = raw_temp[:, 1]

q1, q2, q3 = np.quantile(temp, [0.25, 0.50, 0.75])
iqr = q3 - q1

print(f"기온 데이터 일수: {temp.size}일")
print(f"평균 기온: {temp.mean():.2f} °C")
print(f"중앙값 기온: {np.median(temp):.2f} °C")
print(f"표준편차: {temp.std():.2f}")
print(f"최저/최고: {temp.min():.1f} / {temp.max():.1f} °C")
print(f"Q1: {q1:.2f}, Q2: {q2:.2f}, Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")

### 왜 이 코드가 정답인지

기온 데이터도 걸음 데이터와 같은 2열 구조다. 첫 번째 열은 일자, 두 번째 열은 기온이므로 `raw_temp[:, 1]` 이 분석 대상이다. 평균과 중앙값은 대표값, 표준편차와 IQR은 퍼짐 정도를 보여준다. 이 문제는 같은 NumPy 패턴을 다른 데이터셋에 적용할 수 있는지 확인한다.

### 기대 출력

```
기온 데이터 일수: 90일
평균 기온: 9.67 °C
중앙값 기온: 9.90 °C
표준편차: 5.26
최저/최고: -1.2 / 21.8 °C
Q1: 5.35, Q2: 9.90, Q3: 14.07
IQR: 8.73
```

### 채점 포인트

- `day` 와 `temp_day` 가 같은 1~90 구조라는 점을 확인하면 다음 문제에서 두 데이터를 비교하기 쉽다.
- Q2는 중앙값과 같은 값이어야 한다.

---

## 문제 11 정답 — 기온 조건 필터 만들기

In [ ]:
below_zero_mask = temp < 0
hot_mask = temp > q3
mild_mask = (temp >= 10) & (temp <= 15)

below_zero_days = temp_day[below_zero_mask].tolist()

print(f"영하: {below_zero_mask.sum()}일, 일자 {below_zero_days}")
print(f"Q3 초과: {hot_mask.sum()}일")
print(f"10~15도: {mild_mask.sum()}일")

### 왜 이 코드가 정답인지

각 조건은 90일 기온 배열 전체에 동시에 적용된다. `temp < 0` 은 영하 여부, `temp > q3` 는 상위 25%보다 더운 날 여부를 나타낸다. 10~15도 조건은 두 조건이 동시에 참이어야 하므로 `&` 를 사용한다. 일자 리스트가 필요할 때는 기온 배열이 아니라 `temp_day` 에 마스크를 적용한다.

### 기대 출력

```
영하: 3일, 일자 [2, 8, 9]
Q3 초과: 23일
10~15도: 27일
```

### 자주 보이는 오답

```text
mild_mask = 10 <= temp <= 15
```

파이썬 숫자 하나에는 가능한 표현이지만 NumPy 배열에는 맞지 않는다. 배열에서는 `(temp >= 10) & (temp <= 15)` 로 써야 한다.

---

## 문제 12 정답 — 상품 가격 기본 요약

In [ ]:
prices = np.loadtxt(f"{DATA_BASE}/product_prices.csv", delimiter=",", skiprows=1)

mean_price = prices.mean()
median_price = np.median(prices)
std_price = prices.std()
above_mean = (prices > mean_price).sum()
above_median = (prices > median_price).sum()

print(f"상품 수: {prices.size}개")
print(f"평균 가격: {mean_price:,.0f}원")
print(f"중앙값 가격: {median_price:,.0f}원")
print(f"표준편차: {std_price:,.0f}원")
print(f"최고/최저: {prices.max():,.0f}원 / {prices.min():,.0f}원")
print(f"평균보다 비싼 상품: {above_mean}개")
print(f"중앙값보다 비싼 상품: {above_median}개")

### 왜 이 코드가 정답인지

`product_prices.csv` 는 가격 한 열만 가진 1차원 데이터다. 평균은 전체 가격을 모두 더해 상품 수로 나눈 값이고, 중앙값은 정렬했을 때 가운데 값이다. 평균보다 비싼 상품 수와 중앙값보다 비싼 상품 수를 비교하면 가격 분포가 한쪽으로 치우쳤는지 감을 잡을 수 있다.

### 기대 출력

```
상품 수: 100개
평균 가격: 25,731원
중앙값 가격: 22,250원
표준편차: 16,622원
최고/최저: 86,300원 / 7,600원
평균보다 비싼 상품: 32개
중앙값보다 비싼 상품: 50개
```

### 채점 포인트

- 평균이 중앙값보다 높다는 점은 고가 상품 몇 개가 평균을 끌어올렸다는 해석으로 연결된다.
- `prices > median_price` 는 중앙값보다 큰 상품만 세므로 데이터 수가 짝수일 때 정확히 절반이 나올 수 있다.

---

## 문제 13 정답 — 가격 구간별 상품 수

In [ ]:
under_10k = (prices < 10000).sum()
between_10k_20k = ((prices >= 10000) & (prices < 20000)).sum()
between_20k_40k = ((prices >= 20000) & (prices < 40000)).sum()
over_40k = (prices >= 40000).sum()

total = under_10k + between_10k_20k + between_20k_40k + over_40k

print(f"1만원 미만: {under_10k}개")
print(f"1만원~2만원: {between_10k_20k}개")
print(f"2만원~4만원: {between_20k_40k}개")
print(f"4만원 이상: {over_40k}개")
print(f"합계: {total}개")
assert total == prices.size

### 왜 이 코드가 정답인지

구간별 개수를 셀 때는 각 상품이 정확히 하나의 구간에만 들어가야 한다. 그래서 중간 구간은 아래 경계 이상, 위 경계 미만으로 잡는다. 마지막 `assert` 는 네 구간을 합쳤을 때 전체 상품 수와 일치하는지 확인한다. 이 문제는 등급 분포 문제와 같은 원리를 가격 데이터에 다시 적용하는 연습이다.

### 기대 출력

```
1만원 미만: 8개
1만원~2만원: 35개
2만원~4만원: 45개
4만원 이상: 12개
합계: 100개
```

### 자주 보이는 오답

```text
between_10k_20k = (prices >= 10000) & (prices <= 20000)
between_20k_40k = (prices >= 20000) & (prices <= 40000)
```

20,000원이 두 구간에 동시에 들어갈 수 있다. 이런 중복은 합계 검증에서 드러난다.

---

## 문제 14 정답 — 기온과 걸음 수 관계 보기

In [ ]:
assert np.array_equal(day, temp_day)

corr = np.corrcoef(temp, steps)[0, 1]

if corr > 0.3:
    label = "양의 상관"
elif corr < -0.3:
    label = "음의 상관"
else:
    label = "약한 상관"

print(f"기온-걸음 상관계수: {corr:+.3f}")
print(f"해석: {label}")

### 왜 이 코드가 정답인지

상관계수는 두 숫자 배열이 함께 움직이는 정도를 -1부터 1까지로 요약한다. `np.corrcoef(temp, steps)` 는 2x2 상관행렬을 만들고, `[0, 1]` 위치가 기온과 걸음 수 사이의 상관계수다. `assert np.array_equal(day, temp_day)` 는 두 데이터가 같은 90일 순서로 정렬되어 있는지 확인한다. 날짜가 어긋나면 상관계수를 계산해도 의미가 없다.

### 기대 출력

```
기온-걸음 상관계수: -0.106
해석: 약한 상관
```

### 채점 포인트

- 상관계수가 음수지만 절댓값이 0.3보다 작으므로 강한 음의 상관이라고 해석하면 안 된다.
- 상관관계는 인과관계가 아니다. 이 값만으로 "기온 때문에 덜 걸었다"고 단정하지 않는다.

---

## 문제 15 정답 — 한 페이지 분석 결론 만들기

In [ ]:
score_mean = scores.mean()
score_std = scores.std()
goal_rate = (steps >= 10000).mean() * 100
weekday_weekend_diff = weekday_avg - weekend_avg
corr = np.corrcoef(temp, steps)[0, 1]
price_gap = mean_price - median_price

print(f"점수 평균/표준편차: {score_mean:.2f} / {score_std:.2f}")
print(f"10,000보 달성률: {goal_rate:.1f}%")
print(f"평일-주말 걸음 차이: {weekday_weekend_diff:+,.0f} 보")
print(f"기온-걸음 상관계수: {corr:+.3f}")
print(f"가격 평균-중앙값 차이: {price_gap:,.0f}원")

### 왜 이 코드가 정답인지

결론을 쓰기 전에 핵심 숫자를 다시 모으면 글이 데이터와 어긋나는 것을 막을 수 있다. 점수 평균과 표준편차는 학급 점수의 중심과 퍼짐을 보여준다. 10,000보 달성률과 평일-주말 차이는 활동 패턴을 설명한다. 기온-걸음 상관계수는 두 데이터의 관계가 강한지 약한지 판단하게 한다. 가격 평균-중앙값 차이는 고가 상품이 평균을 끌어올리는지 해석하는 근거가 된다.

### 기대 출력

```
점수 평균/표준편차: 75.61 / 12.39
10,000보 달성률: 26.7%
평일-주말 걸음 차이: +4,007 보
기온-걸음 상관계수: -0.106
가격 평균-중앙값 차이: 3,481원
```

### 결론 예시

## 결론

1. 점수 데이터: 평균은 75.61점이고 표준편차는 12.39점이라 중간권 학생이 많지만, 2표준편차 이상 낮은 학생 7명은 별도 확인이 필요하다.
2. 걸음 데이터: 10,000보 달성률은 26.7%이고 평일이 주말보다 약 4,007보 많아 주말 활동 부족이 가장 뚜렷하다.
3. 기온과 걸음 관계: 상관계수는 -0.106으로 0에 가까워, 이 데이터만 보면 기온이 걸음 수를 강하게 설명한다고 보기 어렵다.
4. 가격 데이터: 평균 가격이 중앙값보다 3,481원 높아 고가 상품 일부가 평균을 끌어올린 것으로 해석할 수 있다.

### 채점 포인트

- 결론은 계산한 숫자와 일치해야 한다.
- "높다/낮다" 같은 말만 쓰고 기준 숫자가 없으면 부분 감점한다.
- LLM으로 그럴듯하게 쓴 문장이라도 코드 출력과 숫자가 다르면 미통과 처리한다.

---

## 전체 채점 메모

| 확인 항목 | 통과 기준 |
|---|---|
| 문제 수 | 학생용 문제 15개, 교사용 정답 15개 |
| 데이터 로드 | `skiprows=1`, `delimiter=","` 사용 |
| 마스크 | `&`, `|`, `~` 사용과 괄호 처리 정확 |
| 검증 | 등급/가격 구간 합계 `assert` 포함 |
| 해석 | 문제 15 결론이 실제 출력 수치와 일치 |
| 학생용 배포 | `solution.ipynb` 는 절대 공유하지 않음 |

문제 1~5는 NumPy 1차원 배열의 기본기를 본다. 문제 6~11은 2열 데이터를 열 단위로 분리하고 조건 마스크를 적용하는 능력을 본다. 문제 12~15는 같은 계산을 가격 데이터와 간단한 관계 해석에 적용하는지 확인한다.

## 문제별 지도 메모

| 문제 | 강사가 볼 핵심 | 학생이 막히면 던질 질문 |
|---:|---|---|
| 1 | CSV 헤더를 건너뛰고 1차원 배열을 만드는지 확인한다. | "에러 메시지에 `score` 라는 글자가 보이나? 그러면 첫 줄을 숫자로 읽으려 한 것이다." |
| 2 | 조건식이 배열 전체에 적용되는지 확인한다. | "`scores >= 90` 을 출력해보면 숫자가 아니라 True/False가 나오지?" |
| 3 | 분위수가 중앙값과 어떻게 연결되는지 확인한다. | "Q2와 문제 1의 중앙값이 같은 이유를 말해볼 수 있나?" |
| 4 | 구간 경계가 겹치지 않는지 확인한다. | "90점 학생은 A와 B 중 어디에만 들어가야 할까?" |
| 5 | 평균에서 멀리 떨어진 값을 양쪽 모두 잡는지 확인한다. | "높은 이상치만 보려는 문제일까, 낮은 이상치도 봐야 할까?" |
| 6 | 2열 배열에서 열 분리가 정확한지 확인한다. | "`raw_steps[:, 0]` 과 `raw_steps[:, 1]` 을 각각 출력하면 무엇이 다른가?" |
| 7 | 마스크를 값 배열과 일자 배열에 모두 적용할 수 있는지 확인한다. | "낮은 활동일의 '걸음 수'가 필요한가, '일자'가 필요한가?" |
| 8 | 슬라이싱 경계가 30일씩 정확히 끊겼는지 확인한다. | "`steps[:30]` 의 길이를 출력하면 몇이어야 할까?" |
| 9 | `&`, `|`, `~` 와 괄호 사용을 확인한다. | "주말 마스크의 True 개수와 평일 마스크의 True 개수를 더하면 90이 되는가?" |
| 10 | 점수 데이터에서 하던 통계 계산을 기온 데이터에 전이하는지 확인한다. | "데이터셋만 바뀌었을 뿐 문제 1, 3과 같은 구조라는 걸 찾을 수 있나?" |
| 11 | 분위수 조건과 범위 조건을 구분하는지 확인한다. | "`temp > q3` 는 상위 몇 퍼센트 정도를 보는 조건일까?" |
| 12 | 평균과 중앙값의 해석 차이를 확인한다. | "평균보다 비싼 상품은 왜 50개가 아니라 더 적을까?" |
| 13 | 가격 구간의 합계 검증을 확인한다. | "구간이 빠지거나 겹치면 `assert` 는 어떻게 반응할까?" |
| 14 | 상관계수의 크기와 부호를 구분하는지 확인한다. | "음수라고 해서 항상 강한 관계라고 말할 수 있을까?" |
| 15 | 숫자와 결론 문장이 일치하는지 확인한다. | "이 결론의 근거가 되는 출력 줄은 어디인가?" |

## 부분 점수 운영 기준

- 출력 형식이 조금 달라도 계산값이 맞고 코드 흐름이 합리적이면 통과 처리한다.
- 숫자 계산은 맞지만 해석이 틀린 경우, 계산 문제는 통과시키고 문제 15 결론에서만 감점한다.
- `np.mean(scores)` 와 `scores.mean()` 은 모두 정답이다. 단, 한 노트북 안에서 스타일이 지나치게 섞이면 정리하도록 안내한다.
- 리스트 컴프리헨션이나 for문으로 푼 학생도 정답 처리할 수 있지만, 이 레슨의 목표가 NumPy 배열 연산이므로 같은 풀이를 NumPy 방식으로 다시 보여준다.
- 학생이 AI로 만든 결론을 붙여 넣은 경우, 코드 출력 숫자와 문장 숫자가 어긋나는지 먼저 확인한다. 숫자가 없거나 데이터에 없는 주장을 하면 다시 쓰게 한다.

## 제출 전 교사용 확인 순서

1. 노트북을 위에서부터 다시 실행해 에러가 남아 있지 않은지 본다.
2. 문제 4와 문제 13의 `assert` 가 통과하는지 본다.
3. 문제 15 결론의 숫자가 문제 1~14 출력과 일치하는지 본다.
4. 학생이 `solution.ipynb` 의 표현을 그대로 베낀 흔적이 있는지 확인한다.
5. 통과 처리 전, 학생에게 문제 2나 문제 9 중 하나를 골라 마스크가 무엇인지 말로 설명하게 한다.